In [ ]:
!pip install -q sentence-transformers faiss-cpu groq langgraph

In [ ]:
sales_records = [
    "South region revenue fell 12% in September 2026.",
    "North region revenue grew 5% in September 2026.",
    "East region revenue was flat in September 2026.",
]

code_commits = [
    "Sept 3 commit: changed the discount rule in the pricing engine.",
    "Aug 20 commit: fixed a checkout bug unrelated to pricing.",
    "Sept 10 commit: refactored the logging module.",
]

policy_docs = [
    "Policy memo: new 15% discount cap approved by finance on Sept 2.",
    "Policy memo: vacation policy updated in July.",
    "Compliance doc: data retention policy unchanged since 2024.",
]

SOURCES = {"sales": sales_records, "code": code_commits, "policy": policy_docs}

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

indexes = {}
for name, records in SOURCES.items():
    embs = embedder.encode(records, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(embs.shape[1])   # normalized vectors -> inner product = cosine similarity
    idx.add(embs)
    indexes[name] = idx

def search(source, query):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, top = indexes[source].search(q, 1)
    return SOURCES[source][top[0][0]]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
FUNCS = {
    "search_sales":  lambda query: search("sales", query),
    "search_code":   lambda query: search("code", query),
    "search_policy": lambda query: search("policy", query),
}

TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
    for name, desc in [
        ("search_sales",  "Search sales and revenue records by region/month."),
        ("search_code",   "Search recent code commits and changes."),
        ("search_policy", "Search policy memos and approval documents."),
    ]
]

In [ ]:
import os, getpass
from groq import Groq
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key:")
client = Groq()
MODEL = "openai/gpt-oss-20b"

Enter your Groq API key:··········


In [ ]:
import json
from langgraph.graph import StateGraph, START, END

SYSTEM_PROMPT = (
    "You answer question using available tools. Call one tool at a time."
    "Once you have enough informtion, answer directly in plain text without calling a tool."
)

In [ ]:
import json
from langgraph.graph import StateGraph, START, END

SYSTEM_PROMPT = (
    "You answer questions using available tools. Call one tool at a time."
    "Once you have enough information, answer directly in plain text without calling a tool."
)

def query_transform(state):
    return state


def think(state):
  resp = client.chat.completions.create(
      model=MODEL, messages=state['messages'], tools=TOOL_SCHEMAS, tool_choice="auto"
  )
  msg = resp.choices[0].message
  messages = state['messages'] + [msg.model_dump(exclude_none=True)]

  if msg.tool_calls:
    tc = msg.tool_calls[0]
    name = tc.function.name
    args = json.loads(tc.function.arguments)
    print(f"THINK: call {name} with {args}")
    return {**state, "messages": messages, "tool_call": tc}

  print("THINK: enoguh info -> finish")
  return {**state, "messages": messages, "tool_call": None, "answer": msg.content}

def act(state):
    tc = state["tool_call"]
    args = json.loads(tc.function.arguments)
    result = FUNCS[tc.function.name](args["query"])
    print(f"ACT:   {tc.function.name} -> {result}")
    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
    return {**state, "messages": state["messages"] + [tool_msg]}

def finish(state):
    print(f"\nFINAL ANSWER: {state['answer']}")
    return state

def route(state):
    return "finish" if state["tool_call"] is None else "act"


graph = StateGraph(dict)


In [ ]:
graph.add_node("think", think)
graph.add_node("act", act)
graph.add_node("finish", finish)
graph.add_node("query_transform", query_transform)
graph.add_edge(START, "query_transform")
graph.add_edge("query_transform", "think")
graph.add_conditional_edges("think", route, {"act": "act", "finish": "finish"})
graph.add_edge("act", "think")
graph.add_edge("finish", END)
app = graph.compile()

In [ ]:
def ask(query):
  print(f"QUERY: {query}\n")
  app.invoke(
      {
          "messages": [
              {"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": query}
          ],
          "tool_call": None,
      }
  )



In [ ]:
ask("Why did South region revenue fall - was there a code change?")

QUERY: Why did South region revenue fall - was there a code change?

THINK: call search_sales with {'query': 'South region revenue fall last quarter'}
ACT:   search_sales -> South region revenue fell 12% in September 2026.
THINK: call search_code with {'query': 'South region code change revenue impact 2026'}
ACT:   search_code -> Sept 3 commit: changed the discount rule in the pricing engine.
THINK: call search_policy with {'query': 'pricing engine discount rule policy South region'}
ACT:   search_policy -> Policy memo: new 15% discount cap approved by finance on Sept 2.
THINK: enoguh info -> finish

FINAL ANSWER: The drop in South‑region revenue was caused by a recent code change.

- **What happened** – On **September 3, 2026** a commit modified the pricing engine to enforce a new 15 % discount cap that had just been approved by finance the day before (Sept 2).  
- **Impact on revenue** – The new cap limits the discount that can be offered to South‑region customers, which reduces the 